In [20]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import nltk
from nltk.corpus import wordnet
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
import string
import re
import nltk
from textblob import TextBlob
from nltk.tokenize.treebank import TreebankWordDetokenizer
import random

In [15]:
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.


True

In [6]:
df_jb = pd.read_csv('/content/drive/MyDrive/promptData/jailbreak_prompts_DAN.csv') #insert path to jailbreak prompt data
df_regular = pd.read_csv('/content/drive/MyDrive/promptData/regular_prompts_DAN.csv') #insert path to regular prompt data
df_jb = df_jb[['prompt', 'jailbreak']]
df_regular = df_regular[['prompt','jailbreak']]
df_jb = df_jb.replace({True:1})
df_regular = df_regular.replace({False:0})
df_jb = df_jb.rename(columns={'jailbreak':'label'})
df_regular = df_regular.rename(columns={'jailbreak':'label'})

In [8]:
df_regular_reduced = df_regular.head(2000)
df_combined = pd.concat([df_jb, df_regular_reduced])
df_combined = df_combined.sample(frac=1).reset_index(drop=True)

In [24]:
X_train, X_test, y_train, y_test = train_test_split(df_combined['prompt'], df_combined['label'], test_size=0.2, random_state=42)
train_df = pd.DataFrame({'prompt': X_train, 'jailbreak': y_train})
test_df = pd.DataFrame({'prompt': X_test, 'jailbreak': y_test})
train_df = train_df.sample(frac=1).reset_index(drop=True)
test_df = test_df.sample(frac=1).reset_index(drop=True)

In [16]:
def preprocess_text(text):
    text = text.lower()
    text = re.sub(r"http\S+|www\S+|https\S+", '', text, flags=re.MULTILINE)
    text = re.sub(r'\@\w+|\#', '', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    tokens = word_tokenize(text)
    stop_words = set(stopwords.words('english'))
    tokens = [word for word in tokens if word not in stop_words]
    text = ' '.join(tokens)
    return text

In [17]:
lemmatizer = WordNetLemmatizer()
def get_wordnet_pos(tag):
    if tag.startswith('J'):
        return wordnet.ADJ
    elif tag.startswith('V'):
        return wordnet.VERB
    elif tag.startswith('N'):
        return wordnet.NOUN
    elif tag.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN
def lemmatize_sentence(sentence):
    words = word_tokenize(sentence)
    pos_tags = nltk.pos_tag(words)
    lemmatized_words = [lemmatizer.lemmatize(word, get_wordnet_pos(tag)) for word, tag in pos_tags]
    lemmatized_sentence = ' '.join(lemmatized_words)
    return lemmatized_sentence

In [18]:
def synonym_replacement(text, num_replacements=2):
    words = text.split()
    new_words = []
    for word in words:
        synonyms = wordnet.synsets(word)
        if synonyms:
            synonym = random.choice(synonyms).lemmas()[0].name()
            if random.uniform(0, 1) < 0.5:
                new_words.append(synonym)
            else:
                new_words.append(word)
        else:
            new_words.append(word)

    return ' '.join(new_words)

In [26]:
test_df['prompt'] = test_df['prompt'].apply(preprocess_text)
test_df['prompt'] = test_df['prompt'].apply(lemmatize_sentence)

In [28]:
train_df['prompt'] = train_df['prompt'].apply(preprocess_text)
train_df['prompt'] = train_df['prompt'].apply(lemmatize_sentence)
synonym_df = train_df.copy()
synonym_df['prompt'] = synonym_df['prompt'].apply(synonym_replacement)

In [29]:
training_data = pd.concat([train_df,synonym_df],ignore_index=True)
training_data = training_data.sample(frac=1, random_state=42).reset_index(drop=True)

In [33]:
training_data.to_csv('/content/training_data_binary.csv', index=False) #set path to save train data
test_df.to_csv('/content/testing_data_binary.csv', index=False) #set path to save test data